# When Graph-Derived Neighbourhood Features Do Not Help: CPU Reproduction

**Paper id:** 366956 · **Mode:** `reproduce` · generated by `paper-to-notebook`

This notebook reproduces the method for paper 366956. Cells are annotated with the paper section they implement.

> Runs on Colab or locally. Check the **Environment** cell first for GPU + deps.

## ⚠️ Manual steps before running

1. Choose a CPU runtime, then run the notebook cells in order.
2. Optional: set RUN_MODE to 'main' before running Configuration for the central CV results.
3. Download the results archive before disconnecting the Colab runtime.

## Environment

In [ ]:
import sys, subprocess, importlib
print('Python', sys.version.split()[0])
try:
    import torch
    print('CUDA available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU:', torch.cuda.get_device_name(0))
except ImportError:
    print('torch not yet installed — see the install cell below')

## Official code

Cloning `https://github.com/VoDucNhatku/graph-neighbourhood-churn-fisat2026`.

In [ ]:
!git clone --depth 1 https://github.com/VoDucNhatku/graph-neighbourhood-churn-fisat2026 repo
%cd repo

## Scope and reference run

Official code: [VoDucNhatku/graph-neighbourhood-churn-fisat2026](https://github.com/VoDucNhatku/graph-neighbourhood-churn-fisat2026), pinned to `fdbbccfc8c0ee91fcefb8f8f3a82f0ee7ada89a4`.

This notebook runs the existing CPU experiments without changing their models, seeds, folds or feature definitions. Select **Runtime > Change runtime type > Hardware accelerator > None**. The optional GPU message in the generated Environment cell is informational; these scripts use CPU libraries. The environment installer creates an isolated Python 3.13.5 interpreter so that the notebook kernel's NumPy packages are not replaced.

The Windows reference run on 2026-10-12 completed all ten jobs in 17 min 42 sec, excluding environment installation. Colab speed has not been measured and may differ. Linux and Windows builds can produce numerical differences despite pinned versions.

Both paths are recorded: the first repeated-CV run builds features from raw XLSX in memory; the second k=10 run reloads the newly created CSV cache. On the audited Windows run, the cached run matched the archived fold CSVs exactly, while the in-memory run differed slightly. Both retained the same null finding.

This is an **as-is reproduction**, not a correction of the paper. Known audit issues remain: midnight date boundaries, L defined as analysis-date minus first purchase, NCR on outgoing kNN sets, full-data SHAP, and insufficient evidence for NCR-only ablation and the cited .398 reproduction claim.

In [ ]:
# Configuration and exact code revision
from pathlib import Path
from datetime import datetime
import subprocess, sys, os, json, hashlib, shutil

RUN_MODE = "full"  # "main": only two 10x5-fold CV runs and the figure; "full": all ten jobs.
COMMIT = "fdbbccfc8c0ee91fcefb8f8f3a82f0ee7ada89a4"
EXPECTED_DATA_SHA256 = "bcbe73b35f5b7babf197fb0cb983a11f5d9ff929078d4aa53d171b1f2df2e980"
REPO = Path.cwd().resolve()  # The generated clone cell above changes into repo/.
assert (REPO / ".git").exists(), "Run the official-code clone cell first."
assert RUN_MODE in ("full", "main")
subprocess.run(["git", "fetch", "--depth", "1", "origin", COMMIT], cwd=REPO, check=True)
subprocess.run(["git", "checkout", "--detach", COMMIT], cwd=REPO, check=True)
assert subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip() == COMMIT
WORK = REPO.parent / "fisat366956"
DATA = WORK / "data"
VENV = WORK / "python313"
RUN = WORK / ("run_" + datetime.now().strftime("%Y%m%d_%H%M%S_%f"))
WORK.mkdir(parents=True, exist_ok=True)
DATA.mkdir(parents=True, exist_ok=True)
print("Code:", COMMIT, "Mode:", RUN_MODE, "Output:", RUN)


## Raw data source *(paper Section 4.1)*

[Online Retail II, UCI dataset 502](https://archive.ics.uci.edu/dataset/502/online%2Bretail%2Bii): 1,067,371 transaction rows across two worksheets, a 43.5 MB XLSX. Citation: Chen, D. (2012), DOI [10.24432/C5CG6D](https://doi.org/10.24432/C5CG6D), **CC BY 4.0**. The download below extracts only the named XLSX and verifies the exact SHA256 used in the Windows run. Raw transaction rows are not included in the output download.

In [ ]:
# Download and fingerprint raw XLSX — paper Section 4.1
import urllib.request, zipfile

DATA_URL = "https://archive.ics.uci.edu/static/public/502/online%2Bretail%2Bii.zip"
xlsx = DATA / "online_retail_II.xlsx"
def file_hash(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()
if not xlsx.exists():
    archive = DATA / "online_retail_II.zip"
    urllib.request.urlretrieve(DATA_URL, archive)
    with zipfile.ZipFile(archive) as package:
        matches = [name for name in package.namelist() if Path(name).name == "online_retail_II.xlsx"]
        assert len(matches) == 1, "Unexpected dataset archive contents."
        with package.open(matches[0]) as source, xlsx.open("wb") as target:
            shutil.copyfileobj(source, target)
actual_hash = file_hash(xlsx)
assert actual_hash == EXPECTED_DATA_SHA256, "Different raw data: stop before training."
print("Dataset:", xlsx.name, "bytes:", xlsx.stat().st_size, "SHA256:", actual_hash)


In [ ]:
# Install the isolated reference CPU environment
import ctypes.util

# uv installs the requested Python without replacing the notebook kernel.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
uv = [sys.executable, "-m", "uv"]
subprocess.run(uv + ["python", "install", "3.13.5"], check=True)
PYTHON = VENV / ("Scripts/python.exe" if os.name == "nt" else "bin/python")
if not PYTHON.exists():
    subprocess.run(uv + ["venv", "--python", "3.13.5", str(VENV)], check=True)
if sys.platform.startswith("linux") and not ctypes.util.find_library("gomp") and shutil.which("apt-get"):
    subprocess.run(["apt-get", "-qq", "update"], check=True)
    subprocess.run(["apt-get", "-qq", "install", "-y", "libgomp1"], check=True)
subprocess.run(uv + ["pip", "install", "--python", str(PYTHON), "-r", str(REPO / "requirements.txt")], check=True)
subprocess.run([str(PYTHON), "-c", "import platform, lightgbm, numpy; print('Reference Python', platform.python_version(), 'LightGBM', lightgbm.__version__, 'NumPy', numpy.__version__)"], check=True)


In [ ]:
# Write the portable experiment runner
RUNNER_SOURCE = '"""Run the unchanged paper code on a pinned CPU environment, then compare outputs.\n\nUsage: python reproduce_366956.py --repo PATH --data PATH --out NEW_PATH --mode full\nThe notebook embeds this entire file; a separate upload of the helper is unnecessary.\n"""\nfrom pathlib import Path\nfrom datetime import datetime\nfrom importlib import metadata\nimport argparse\nimport ast\nimport hashlib\nimport json\nimport os\nimport platform\nimport shutil\nimport subprocess\nimport sys\nimport time\n\nCOMMIT = "fdbbccfc8c0ee91fcefb8f8f3a82f0ee7ada89a4"\nDATA_SHA256 = "bcbe73b35f5b7babf197fb0cb983a11f5d9ff929078d4aa53d171b1f2df2e980"\nPAPER_TABLE1 = {\n    "RFM": (.8066, None, None), "LRFM": (.8074, .0009, .755),\n    "B-LRFM": (.8054, -.0012, .742), "B-LRFM+Seg": (.8044, -.0022, .561),\n    "Graph-Enhanced": (.8044, -.0021, .590),\n}\n\n\ndef sha256(path):\n    return hashlib.sha256(Path(path).read_bytes()).hexdigest()\n\n\ndef job_plan(mode):\n    initial = [("pipeline", "run_pipeline_v2.py", "main", {}),\n               ("comparison", "compare_methods_v2.py", "main", {}),\n               ("auxiliary", "improvements_pipeline.py", "main", {})] if mode == "full" else []\n    degrees = [5, 10, 15, 20] if mode == "full" else [10]\n    jobs = initial + [("repeated_cv_raw", "significance_test.py", "main", {})]\n    jobs += [(f"knn_k{k}", "significance_test.py", "main",\n              {"LRFM_KNN_K": str(k), "LRFM_OUT_SUFFIX": f"_k{k}"}) for k in degrees]\n    if mode == "full":\n        jobs.append(("seed42", "significance_test.py", "seed42", {"LRFM_CV_SEEDS": "42"}))\n    return jobs + [("figure", "make_paper_figures.py", "main", {})]\n\n\ndef analyze(repo, run):\n    import numpy as np\n    import pandas as pd\n    from scipy import stats\n    repo, run = Path(repo), Path(run)\n    output = run / "outputs/main"\n    raw = pd.read_csv(output / "significance_summary.csv")\n    cached = pd.read_csv(output / "significance_summary_k10.csv")\n    rows = []\n    for tier, (auc, delta, p) in PAPER_TABLE1.items():\n        a = raw.query("Metric==\'AUC\' and Model==\'LightGBM\' and Tier==@tier").iloc[0]\n        b = cached.query("Metric==\'AUC\' and Model==\'LightGBM\' and Tier==@tier").iloc[0]\n        rows.append({"Tier": tier, "Paper_AUC": auc, "Raw_AUC": float(a.Mean),\n                     "Cached_k10_AUC": float(b.Mean), "Paper_delta": delta,\n                     "Raw_delta": None if delta is None else float(a.Delta_vs_RFM),\n                     "Cached_k10_delta": None if delta is None else float(b.Delta_vs_RFM),\n                     "Paper_pNB": p, "Raw_pNB": None if p is None else float(a.p_NadeauBengio),\n                     "Cached_k10_pNB": None if p is None else float(b.p_NadeauBengio),\n                     "Cached_matches_paper_rounding": bool(abs(b.Mean-auc) <= .00005+1e-12\n                         and (p is None or abs(b.p_NadeauBengio-p) <= .0005+1e-12)\n                         and (delta is None or abs(b.Delta_vs_RFM-delta) <= .00005+1e-12))})\n    pd.DataFrame(rows).to_csv(run / "paper-table1-comparison.csv", index=False)\n\n    archive = []\n    candidates = [(p.name, repo / "results" / p.name, p) for p in sorted(output.glob("*.csv"))]\n    candidates += [("cached-k10-as-published/"+name, repo / "results" / name,\n                    output / name.replace(".csv", "_k10.csv"))\n                   for name in ["significance_folds.csv", "significance_summary.csv"]]\n    for label, old_path, new_path in candidates:\n        if not old_path.exists():\n            continue\n        old, new = pd.read_csv(old_path), pd.read_csv(new_path)\n        same_shape = old.shape == new.shape and old.columns.tolist() == new.columns.tolist()\n        equal, peak = same_shape, 0.0\n        if same_shape:\n            for column in old.columns:\n                if pd.api.types.is_numeric_dtype(old[column]) and pd.api.types.is_numeric_dtype(new[column]):\n                    a, b = old[column].to_numpy(float), new[column].to_numpy(float)\n                    equal &= bool(np.isclose(a, b, atol=1e-12, rtol=0, equal_nan=True).all())\n                    finite = np.isfinite(a) & np.isfinite(b)\n                    if finite.any():\n                        peak = max(peak, float(np.max(np.abs(a[finite]-b[finite]))))\n                else:\n                    equal &= old[column].fillna("<missing>").astype(str).equals(\n                        new[column].fillna("<missing>").astype(str))\n        archive.append({"File": label, "Byte_identical": old_path.read_bytes() == new_path.read_bytes(),\n                        "Same_shape": same_shape, "Equal_at_1e12": bool(equal), "Max_numeric_delta": peak})\n    pd.DataFrame(archive).to_csv(run / "archive-comparison.csv", index=False)\n\n    module = ast.parse((repo / "src/significance_test.py").read_text(encoding="utf-8"))\n    functions = [n for n in module.body if isinstance(n, ast.FunctionDef)\n                 and n.name in ["nadeau_bengio_t", "bootstrap_ci"]]\n    namespace = {"np": np, "stats": stats, "SEED": 42}\n    exec(compile(ast.Module(body=functions, type_ignores=[]), "<paper-statistics>", "exec"), namespace)\n    integrity = []\n    for folds_path in sorted(output.glob("significance_folds*.csv")):\n        folds = pd.read_csv(folds_path)\n        summary = pd.read_csv(output / folds_path.name.replace("folds", "summary"))\n        assert len(folds) == 500 and not folds.duplicated(["FoldId", "Tier", "Model"]).any()\n        assert folds.groupby(["Tier", "Model"]).size().eq(50).all()\n        max_error = 0.0\n        for metric in ["AUC", "PR_AUC"]:\n            for model in ["LightGBM", "Logistic Regression"]:\n                pivot = folds.query("Model==@model").pivot(index="FoldId", columns="Tier", values=metric)\n                for tier in pivot.columns:\n                    row = summary.query("Metric==@metric and Model==@model and Tier==@tier").iloc[0]\n                    max_error = max(max_error, abs(float(row.Mean)-float(pivot[tier].mean())))\n                    if tier != "RFM":\n                        _, p = namespace["nadeau_bengio_t"]((pivot[tier]-pivot.RFM).to_numpy(), 5)\n                        max_error = max(max_error, abs(p-float(row.p_NadeauBengio)))\n        assert max_error < 1e-12\n        integrity.append({"File": folds_path.name, "Rows": len(folds), "Max_statistical_error": max_error})\n\n    result = {"table1": rows, "archive": archive, "integrity": integrity,\n              "no_significant_tier_vs_rfm_raw": bool(raw.query("Tier!=\'RFM\'").p_NadeauBengio.ge(.05).all()),\n              "no_significant_tier_vs_rfm_cached_k10": bool(cached.query("Tier!=\'RFM\'").p_NadeauBengio.ge(.05).all()),\n              "scope": "Unchanged as-is code; this does not resolve methodological discrepancies in the manuscript."}\n    (run / "comparison.json").write_text(json.dumps(result, indent=2, allow_nan=False), encoding="utf-8")\n    print(pd.DataFrame(rows).to_string(index=False), flush=True)\n    return result\n\n\ndef execute(repo, data, run, mode):\n    repo, data, run = Path(repo).resolve(), Path(data).resolve(), Path(run).resolve()\n    if run.exists():\n        raise SystemExit("Choose a new output directory: the first CV run must start without a feature cache.")\n    actual_commit = subprocess.check_output(["git", "-C", str(repo), "rev-parse", "HEAD"], text=True).strip()\n    if actual_commit != COMMIT:\n        raise SystemExit(f"Unexpected code revision: {actual_commit}")\n    if platform.python_version() != "3.13.5":\n        raise SystemExit("Run with the isolated Python 3.13.5 interpreter created in the notebook.")\n    requirements = {}\n    for line in (repo / "requirements.txt").read_text(encoding="utf-8").splitlines():\n        if "==" in line and not line.lstrip().startswith("#"):\n            name, version = line.strip().split("==")\n            installed = metadata.version(name)\n            if installed != version:\n                raise SystemExit(f"Dependency mismatch: {name} {installed}, expected {version}")\n            requirements[name] = installed\n    if sha256(data / "online_retail_II.xlsx") != DATA_SHA256:\n        raise SystemExit("Dataset hash does not match the audited raw XLSX.")\n    original_sources = {p.name: sha256(p) for p in (repo / "src").glob("*.py")}\n    run.mkdir(parents=True)\n    snapshot = run / "snapshot"\n    for folder in ["src", "paper"]:\n        shutil.copytree(repo / folder, snapshot / folder)\n    logs = run / "logs"\n    logs.mkdir()\n    manifest = {"started": datetime.now().astimezone().isoformat(), "status": "running",\n                "mode": mode, "commit": COMMIT, "dataset_sha256": DATA_SHA256,\n                "python": sys.version, "platform": platform.platform(), "logical_cpus": os.cpu_count(),\n                "requirements": requirements, "source_sha256": original_sources, "jobs": []}\n    def save():\n        (run / "run-manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")\n    save()\n    for name, script, output_name, overrides in job_plan(mode):\n        output = run / "outputs" / output_name\n        output.mkdir(parents=True, exist_ok=True)\n        if name == "seed42":\n            shutil.copy2(run / "outputs/main/feat_cache.csv", output / "feat_cache.csv")\n        env = os.environ.copy()\n        for key in ["LRFM_CV_SEEDS", "LRFM_KNN_K", "LRFM_OUT_SUFFIX"]:\n            env.pop(key, None)\n        env.update({"LRFM_DATA_DIR": str(data), "LRFM_OUT_DIR": str(output),\n                    "PYTHONIOENCODING": "utf-8", "PYTHONUNBUFFERED": "1", "MPLBACKEND": "Agg"})\n        env.update(overrides)\n        record = {"name": name, "script": script, "overrides": overrides, "status": "running"}\n        manifest["jobs"].append(record)\n        save()\n        print(f"START {name}; full output: {logs / (name+\'.log\')}", flush=True)\n        started = time.perf_counter()\n        with (logs / (name+".log")).open("w", encoding="utf-8") as log:\n            process = subprocess.run([sys.executable, "-u", str(snapshot / "src" / script)],\n                                     cwd=snapshot, env=env, stdout=log, stderr=subprocess.STDOUT)\n        record.update({"exit_code": process.returncode, "seconds": time.perf_counter()-started,\n                       "status": "completed" if process.returncode == 0 else "failed"})\n        print(f"END {name}: exit={process.returncode}, seconds={record[\'seconds\']:.1f}", flush=True)\n        save()\n        if process.returncode:\n            manifest["status"] = "failed"\n            save()\n            print((logs / (name+".log")).read_text(encoding="utf-8")[-5000:])\n            raise SystemExit(f"Experiment {name} failed; inspect its log.")\n    analyze(repo, run)\n    manifest.update({"status": "completed", "finished": datetime.now().astimezone().isoformat(),\n                     "dataset_unchanged": sha256(data / "online_retail_II.xlsx") == DATA_SHA256,\n                     "original_sources_unchanged": all(sha256(repo / "src" / name) == value\n                                                        for name, value in original_sources.items())})\n    save()\n    print(f"Completed. Results, comparison, logs and manifest: {run}", flush=True)\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--repo", required=True, type=Path)\n    parser.add_argument("--data", type=Path)\n    parser.add_argument("--out", required=True, type=Path)\n    parser.add_argument("--mode", choices=["full", "main"], default="full")\n    parser.add_argument("--analyze-only", action="store_true")\n    args = parser.parse_args()\n    if args.analyze_only:\n        analyze(args.repo, args.out)\n    elif args.data is None:\n        parser.error("--data is required when running experiments")\n    else:\n        execute(args.repo, args.data, args.out, args.mode)\n'
RUNNER = WORK / 'reproduce_366956.py'
RUNNER.write_text(RUNNER_SOURCE, encoding='utf-8')
print('Runner:', RUNNER)


## Run unchanged experiments *(paper Sections 5-8)*

`full` runs the pipeline (communities, classifiers, SHAP), clustering comparisons, auxiliary variants, 10x5-fold CV from raw features, k=5/10/15/20 with the freshly built cache, a seed-42 CV run and the results figure. `main` runs raw 10x5-fold CV, cached k=10 10x5-fold CV and the figure. It does not reduce seeds or folds.

Each job writes a log; the manifest records versions, hashes, job time and return code. All new outputs are in the fresh timestamped directory. The original GitHub result CSVs remain available for comparison. The helper refuses to start if this output directory already exists.

The auxiliary variants and SHAP retain their original full-data label aggregation; they are not evidence for the controlled repeated-CV finding. Dataset construction is transductive as in the original code.

In [ ]:
# Execute training and statistical comparison — paper Sections 7-8
command = [str(PYTHON), "-u", str(RUNNER), "--repo", str(REPO), "--data", str(DATA),
           "--out", str(RUN), "--mode", RUN_MODE]
with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                      text=True, encoding="utf-8", bufsize=1) as process:
    for line in process.stdout:
        print(line, end="", flush=True)
    return_code = process.wait()
assert return_code == 0, "Runner failed: inspect the displayed output and job logs."
manifest = json.loads((RUN / "run-manifest.json").read_text(encoding="utf-8"))
assert manifest["status"] == "completed"
assert manifest["dataset_unchanged"] and manifest["original_sources_unchanged"]
print("All requested jobs completed:", len(manifest["jobs"]))


In [ ]:
# Display paper versus raw and cached results — paper Table 1 and Section 8.3
import csv
from IPython.display import display, HTML
from html import escape

def show_csv(path):
    with Path(path).open(encoding="utf-8", newline="") as source:
        rows = list(csv.reader(source))
    table = "<table><thead><tr>" + "".join("<th>"+escape(v)+"</th>" for v in rows[0]) + "</tr></thead><tbody>"
    table += "".join("<tr>"+"".join("<td>"+escape(v)+"</td>" for v in row)+"</tr>" for row in rows[1:])
    display(HTML(table+"</tbody></table>"))
show_csv(RUN / "paper-table1-comparison.csv")
show_csv(RUN / "archive-comparison.csv")
comparison = json.loads((RUN / "comparison.json").read_text(encoding="utf-8"))
print("No significant tier-vs-RFM tests (raw):", comparison["no_significant_tier_vs_rfm_raw"])
print("No significant tier-vs-RFM tests (cached k10):", comparison["no_significant_tier_vs_rfm_cached_k10"])
print("Fold integrity:", comparison["integrity"])


In [ ]:
# Download results and logs
# Export aggregate metrics, logs, manifest, source snapshot and the figure.
# Exclude per-customer cache and segment assignments from the download.
import zipfile
archive_path = RUN.parent / (RUN.name + "_results.zip")
with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_DEFLATED) as package:
    for path in sorted(RUN.rglob("*")):
        if not path.is_file() or "__pycache__" in path.parts:
            continue
        if path.name == "feat_cache.csv" or path.suffix == ".xlsx" or "assignment" in path.name.lower():
            continue
        package.write(path, path.relative_to(RUN))
print("Results archive:", archive_path)
try:
    from google.colab import files
except ImportError:
    print("Local Jupyter: download the archive from the path above.")
else:
    files.download(str(archive_path))
